## **YOLOv8-Seg con Kvasir-SEG**

En este fichero se procede a evaluar el rendimiento de la arquitectura YOLOv8-Seg en la segmentación de pólipos usando el dataset de Kvasir-SEG.

En este primer bloque de código se importan las librerías necesarias para la ejecución del fichero completo. Además, se importan las funciones de métricas necesarias, para las cuales se tuvo que añadir la raíz del proyecto al path de Python para que los imports funcionasen correctamente. Finalmente, se define la ruta al dataset con el que entrenaremos los modelos.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
from ultralytics import YOLO
import segmentation_models_pytorch as smp
import numpy as np
import os
import cv2
import pandas as pd
import time
import sys
import torch
import shutil


root_path = os.path.abspath('..')
if root_path not in sys.path:
    sys.path.append(root_path)

from utils.segmentation_quality_metrics import boundary_iou, hausdorff_95, compute_all_metrics

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Kvasir-SEG"


La siguiente función convierte una máscara binaria al formato de polígono requerido por YOLO-Seg. Para ello, binariza la máscara con un umbral de 127 y extrae sus contornos externos con OpenCV. Si no hay contornos o el contorno principal tiene menos de tres puntos, devuelve None, ya que no se puede formar un polígono con menos de tres puntos. En caso contrario, selecciona el contorno de mayor área, que corresponde al objeto principal, y normaliza sus coordenadas dividiéndolas entre el ancho y el alto de la imagen. De esta manera, se genera una lista de valores entre cero y uno que es el formato que espera YOLO para los polígonos de segmentación.

In [10]:
def mask_to_yolo_polygon(mask_path, img_w, img_h):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask = (mask > 127).astype(np.uint8)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        return None
    contour = max(contours, key=cv2.contourArea)
    if len(contour) < 3:
        return None
    points = contour.squeeze()
    if points.ndim == 1:
        return None
    normalized = []
    for x, y in points:
        normalized.extend([x / img_w, y / img_h])
    return normalized


La función convert_kvasir_to_yolo recorre los tres splits del dataset y genera un fichero .txt por cada imagen con el polígono de segmentación en formato YOLO. Para cada imagen obtiene sus dimensiones reales, llama a la función mask_to_yolo_polygon para extraer el contorno normalizado y lo escribe en el fichero de etiquetas con el identificador de clase 0, que corresponde a la única clase del dataset (pólipo). Las etiquetas se guardan en una carpeta labels dividida en tres splits (train, val y test).

A continuación, se genera el fichero dataset_yolo.yaml, que es el fichero de configuración que necesita YOLO para localizar las imágenes y etiquetas de cada split, el número de clases y sus nombres.

In [ ]:
def convert_kvasir_to_yolo(dataset_path):
    for split in ["train", "val", "test"]:
        images_dir = os.path.join(dataset_path, "images", split)
        masks_dir  = os.path.join(dataset_path, "masks",  split)
        labels_dir = os.path.join(dataset_path, "labels", split)
        os.makedirs(labels_dir, exist_ok=True)

        for img_file in os.listdir(images_dir):
            if not img_file.endswith(".jpg"):
                continue
            name = os.path.splitext(img_file)[0]
            mask_path = os.path.join(masks_dir, name + ".jpg")
            if not os.path.exists(mask_path):
                continue

            img = cv2.imread(os.path.join(images_dir, img_file))
            h, w = img.shape[:2]

            polygon = mask_to_yolo_polygon(mask_path, w, h)
            if polygon is None:
                continue

            label_path = os.path.join(labels_dir, name + ".txt")
            with open(label_path, "w") as f:
                f.write("0 " + " ".join(f"{v:.6f}" for v in polygon) + "\n")

convert_kvasir_to_yolo(dataset)

yaml_content = f"""path: {dataset}
train: images/train
val: images/val
test: images/test

nc: 1
names: ['polyp']
"""

yaml_path = os.path.join(dataset, "dataset_yolo.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content)


La función contenida en la siguiente celda instancia YOLOv8n-Seg con pesos preentrenados en COCO y lanza el entrenamiento sobre Kvasir-SEG durante 50 épocas con imágenes de 512x512 y batch size de 4. YOLO gestiona internamente el bucle de entrenamiento, la validación y el guardado de checkpoints (que consiste en el guardado de los mejores resultados hasta ese momento), por lo que no es necesario implementarlo manualmente. Finalmente, copia los mejores pesos, que YOLO selecciona automáticamente según la métrica de validación, a la ruta de salida especificada y los devuelve para usarlos durante la evaluación.

In [ ]:
def train_yolo(yaml_path, output_weights, epochs=50):
    model = YOLO("yolov8n-seg.pt")
    model.train(
        data=yaml_path,
        epochs=epochs,
        imgsz=512,
        batch=4,
        device=0,
        verbose=False
    )
    best_weights = model.trainer.best
    shutil.copy(best_weights, output_weights)
    return output_weights

La función evaluate_yolo carga los pesos entrenados y evalúa el modelo sobre el conjunto de test. Cabe destacar que YOLO genera múltiples detecciones por imagen, por lo que se selecciona la máscara con mayor puntuación de confianza como predicción final. Si YOLO no detecta ningún objeto en una imagen, esta se descarta. La máscara predicha se redimensiona a las dimensiones originales para poder compararla con el ground truth, y se miden la latencia y la VRAM en cada inferencia. Finalmente, se devuelve un diccionario con la media de cada métrica sobre todo el conjunto de test.

Por último, se lanzan el entrenamiento y la evaluación y se miden sus tiempos. Estos tiempos se añaden al diccionario de resultados, que finalmente se guarda en un CSV. Si el fichero ya existe se añade una nueva línea, sin sobreescribir los resultados previos para poder comparar los resultados entre distintos experimentos.

In [ ]:
def evaluate_yolo(dataset_path, weights_path):
    model = YOLO(weights_path)

    test_images_dir = os.path.join(dataset_path, "images", "test")
    test_masks_dir  = os.path.join(dataset_path, "masks",  "test")

    iou_scores, precision_scores, recall_scores, f1_scores = [], [], [], []
    dice_scores, specificity_scores, f2_scores, pixel_accuracy_scores = [], [], [], []
    boundary_iou_scores, hausdorff_95_scores, latency_scores, vram_scores = [], [], [], []

    for img_file in os.listdir(test_images_dir):
        if not img_file.endswith(".jpg"):
            continue
        name = os.path.splitext(img_file)[0]
        img_path  = os.path.join(test_images_dir, img_file)
        mask_path = os.path.join(test_masks_dir, name + ".jpg")
        if not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)
        H, W = gt_mask.shape

        if torch.cuda.is_available():
            vram_before = torch.cuda.memory_allocated() / 1024**2

        start = time.time()
        results = model(img_path, verbose=False)
        latency = (time.time() - start) * 1000

        if torch.cuda.is_available():
            vram = torch.cuda.memory_allocated() / 1024**2 - vram_before
        else:
            vram = 0

        if results[0].masks is None or len(results[0].masks.data) == 0:
            continue

        scores = results[0].boxes.conf.cpu().numpy()
        best_idx = np.argmax(scores)
        pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(np.uint8)
        pred_mask = cv2.resize(pred_mask, (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    results = {
        "modelo": ["yolov8n_seg_kvasir"],
        "mean_iou": [np.mean(iou_scores)],
        "mean_f1": [np.mean(f1_scores)],
        "mean_recall": [np.mean(recall_scores)],
        "mean_precision": [np.mean(precision_scores)],
        "mean_dice": [np.mean(dice_scores)],
        "mean_specificity": [np.mean(specificity_scores)],
        "mean_f2": [np.mean(f2_scores)],
        "mean_pixel_accuracy": [np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou": [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95": [np.mean(hausdorff_95_scores)],
        "mean_latency_ms": [np.mean(latency_scores)],
        "mean_vram_mb": [np.mean(vram_scores)]
    }

    return results


output_weights_yolo = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_finetuned\\pesos_yolov8n_seg_kvasir.pt"
output_path_yolo = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_yolo.csv"

start_train = time.time()
trained_weights = train_yolo(yaml_path, output_weights_yolo)
train_time = time.time() - start_train

start_eval = time.time()
results = evaluate_yolo(dataset, trained_weights)
eval_time = time.time() - start_eval

results["train_time_minutes"] = [round(train_time / 60, 2)]
results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
if os.path.exists(output_path_yolo):
    df.to_csv(output_path_yolo, mode='a', header=False, index=False)
else:
    df.to_csv(output_path_yolo, index=False)
